# 🧑‍💼 Glassdoor Jobs — Salary Analysis & Prediction
**Dataset:** [Jobs Dataset from Glassdoor](https://www.kaggle.com/datasets/thedevastator/jobs-dataset-from-glassdoor)

This notebook covers:
1. Data Loading & Exploration
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Salary Prediction Model (Random Forest)
5. Model Export for Flask Backend


## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install any missing libs (Colab usually has these)
!pip install -q scikit-learn matplotlib seaborn pandas numpy joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Libraries loaded!')

## 📂 Step 2 — Upload & Load Data
> Upload `eda_data.csv`, `glassdoor_jobs.csv`, and `salary_data_cleaned.csv` when prompted.

In [ ]:
from google.colab import files
print('Upload your CSV files now:')
uploaded = files.upload()

In [ ]:
# Load all three datasets
eda_df      = pd.read_csv('eda_data.csv')
glass_df    = pd.read_csv('glassdoor_jobs.csv')
salary_df   = pd.read_csv('salary_data_cleaned.csv')

print(f'eda_data       : {eda_df.shape}')
print(f'glassdoor_jobs : {glass_df.shape}')
print(f'salary_cleaned : {salary_df.shape}')
eda_df.head()

## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== EDA Data Info ===')
eda_df.info()
print('\n=== Missing Values ===')
print(eda_df.isnull().sum()[eda_df.isnull().sum() > 0])

In [ ]:
print('=== Salary Stats (avg_salary in $K) ===')
print(eda_df['avg_salary'].describe())

In [ ]:
# --- Plot 1: Salary Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(eda_df['avg_salary'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Average Salary ($K)', fontsize=13)
axes[0].set_xlabel('Avg Salary ($K)')
axes[0].set_ylabel('Count')

axes[1].boxplot(eda_df['avg_salary'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Salary Boxplot', fontsize=13)
axes[1].set_ylabel('Avg Salary ($K)')

plt.tight_layout()
plt.savefig('salary_distribution.png', dpi=150)
plt.show()
print('📊 Plot saved as salary_distribution.png')

In [ ]:
# --- Plot 2: Avg Salary by Job Role ---
job_salary = eda_df[eda_df['job_simp'] != 'na'].groupby('job_simp')['avg_salary'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(job_salary.index, job_salary.values, color=sns.color_palette('Blues_r', len(job_salary)))
plt.title('Average Salary by Job Role ($K)', fontsize=14)
plt.xlabel('Job Role')
plt.ylabel('Avg Salary ($K)')
for bar, val in zip(bars, job_salary.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'${val:.0f}K',
             ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('salary_by_role.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 3: Avg Salary by Sector (Top 10) ---
sector_salary = eda_df.groupby('Sector')['avg_salary'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=sector_salary.index, y=sector_salary.values, palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Sectors by Average Salary ($K)', fontsize=14)
plt.ylabel('Avg Salary ($K)')
plt.tight_layout()
plt.savefig('salary_by_sector.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 4: Skills Demand ---
skills = {'Python': eda_df['python_yn'].sum(),
          'R':      eda_df['R_yn'].sum(),
          'Spark':  eda_df['spark'].sum(),
          'AWS':    eda_df['aws'].sum(),
          'Excel':  eda_df['excel'].sum()}

plt.figure(figsize=(8, 5))
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']
plt.bar(skills.keys(), skills.values(), color=colors, edgecolor='white')
plt.title('Skills Mentioned in Job Postings', fontsize=14)
plt.ylabel('Number of Jobs')
for i, (k, v) in enumerate(skills.items()):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('skills_demand.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 5: Salary Heatmap (Skills vs Job Role) ---
role_skill = eda_df[eda_df['job_simp'] != 'na'].groupby('job_simp')[['python_yn','spark','aws','excel']].mean() * 100

plt.figure(figsize=(9, 5))
sns.heatmap(role_skill, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': '% of Jobs'})
plt.title('Skill Prevalence (%) by Job Role', fontsize=13)
plt.tight_layout()
plt.savefig('skills_heatmap.png', dpi=150)
plt.show()

In [ ]:
# --- Plot 6: Top 10 States by Job Count ---
state_counts = eda_df['job_state'].value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=state_counts.index, y=state_counts.values, palette='coolwarm')
plt.title('Top 10 States by Data Science Job Postings', fontsize=13)
plt.ylabel('Job Count')
plt.tight_layout()
plt.savefig('jobs_by_state.png', dpi=150)
plt.show()

## 🤖 Step 4 — Machine Learning: Salary Prediction

In [ ]:
# Feature Engineering
df = eda_df.copy()

# Encode categorical features
le = LabelEncoder()
df['job_simp_enc'] = le.fit_transform(df['job_simp'].fillna('na'))
df['seniority_enc'] = le.fit_transform(df['seniority'].fillna('na'))
df['sector_enc'] = le.fit_transform(df['Sector'].fillna('Unknown'))
df['size_enc'] = le.fit_transform(df['Size'].fillna('Unknown'))
df['state_enc'] = le.fit_transform(df['job_state'].fillna('Unknown'))

# Feature columns
FEATURES = ['python_yn', 'R_yn', 'spark', 'aws', 'excel',
            'Rating', 'age', 'desc_len', 'num_comp',
            'same_state', 'hourly', 'employer_provided',
            'job_simp_enc', 'seniority_enc', 'sector_enc',
            'size_enc', 'state_enc']

TARGET = 'avg_salary'

ml_df = df[FEATURES + [TARGET]].dropna()
X = ml_df[FEATURES]
y = ml_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Train Models
models = {
    'Linear Regression':     LinearRegression(),
    'Random Forest':         RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingRegressor(n_estimators=200, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2  = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'R2': r2, 'model': model}
    print(f'{name:25s} → MAE: ${mae:.1f}K  |  R²: {r2:.3f}')

In [ ]:
# Pick best model by R2
best_name = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'\n🏆 Best Model: {best_name} (R²={results[best_name]["R2"]:.3f})')

# Feature Importance (if tree-based)
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    fi.plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title(f'Feature Importance — {best_name}', fontsize=13)
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150)
    plt.show()

In [ ]:
# Actual vs Predicted Plot
preds = best_model.predict(X_test)
plt.figure(figsize=(7, 6))
plt.scatter(y_test, preds, alpha=0.5, color='steelblue', edgecolors='white', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Salary ($K)')
plt.ylabel('Predicted Salary ($K)')
plt.title(f'Actual vs Predicted — {best_name}', fontsize=13)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()

## 💾 Step 5 — Export Model & Data for Flask Backend

In [ ]:
import json

# Save model
joblib.dump(best_model, 'salary_model.pkl')
print('✅ Model saved: salary_model.pkl')

# Save feature list
with open('model_features.json', 'w') as f:
    json.dump(FEATURES, f)
print('✅ Features saved: model_features.json')

# Save summary stats for dashboard
summary = {
    'avg_salary_mean': float(eda_df['avg_salary'].mean()),
    'avg_salary_median': float(eda_df['avg_salary'].median()),
    'total_jobs': int(len(eda_df)),
    'salary_by_role': eda_df[eda_df['job_simp'] != 'na'].groupby('job_simp')['avg_salary'].mean().round(1).to_dict(),
    'salary_by_sector': eda_df.groupby('Sector')['avg_salary'].mean().sort_values(ascending=False).head(10).round(1).to_dict(),
    'skills': {
        'Python': int(eda_df['python_yn'].sum()),
        'R':      int(eda_df['R_yn'].sum()),
        'Spark':  int(eda_df['spark'].sum()),
        'AWS':    int(eda_df['aws'].sum()),
        'Excel':  int(eda_df['excel'].sum())
    },
    'top_states': eda_df['job_state'].value_counts().head(10).to_dict()
}
with open('dashboard_data.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('✅ Dashboard data saved: dashboard_data.json')

In [ ]:
# Download all output files
from google.colab import files
for fname in ['salary_model.pkl', 'model_features.json', 'dashboard_data.json',
              'salary_distribution.png', 'salary_by_role.png', 'salary_by_sector.png',
              'skills_demand.png', 'skills_heatmap.png', 'jobs_by_state.png',
              'feature_importance.png', 'actual_vs_predicted.png']:
    try:
        files.download(fname)
    except Exception as e:
        print(f'Could not download {fname}: {e}')

print('\n🎉 All done! Download these files and put them in your Flask backend/static/ folder.')

---
## ✅ Next Steps
1. Download `salary_model.pkl`, `model_features.json`, `dashboard_data.json`
2. Place them in your VS Code project: `backend/` folder
3. Run `pip install -r requirements.txt` in VS Code
4. Run `python app.py` → open `http://localhost:5000`
